In [20]:
import pandas as pd
import numpy as np

import glob

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder

In [21]:
train = pd.read_csv("datasets/train-processed.csv")
test = pd.read_csv("datasets/test_split.csv")

In [22]:
# --- 2. Preparação (Data Preparation) ---

# 1. Definindo features e target usando o dataframe 'train'
X = train.drop(columns=['Taxa de Congestionamento_mes (%)'])
y = train['Taxa de Congestionamento_mes (%)']

# 2. Divisão Treino/Validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [23]:
# --- 3. Construção do Pipeline Robusto ---

# A estratégia aqui é usar apenas a 'serventia' via Target Encoding.
# A 'comarca' pode ser removida pois a serventia captura a granularidade,
# ou mantida se você acreditar que a comarca adiciona um "viés regional" distinto.
# Vamos manter ambas para demonstrar como o Ridge lida com a correlação.

cols_para_encode = ['comarca', 'serventia']

# Pipeline de processamento das features
model_pipeline = Pipeline([
    # Passo 1: Target Encoding
    # min_samples_leaf: Mínimo de amostras para levar a média da categoria em conta
    # smoothing: Quanto maior, mais a média da categoria é puxada para a média global (regularização do encoder)
    ('encoder', TargetEncoder(cols=cols_para_encode, min_samples_leaf=20, smoothing=10)),
    
    # Passo 2: Padronização (Obrigatório para Regressão Regularizada como Ridge/Lasso)
    ('scaler', StandardScaler()),
    
    # Passo 3: Modelo Ridge com Cross-Validation interno para achar o melhor alpha (força da regularização)
    ('regressor', RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], scoring='neg_mean_squared_error'))
])


In [24]:
# --- 4. Treinamento e Avaliação ---

X_test = test.drop(columns=['Taxa de Congestionamento_mes (%)'])
y_test = test['Taxa de Congestionamento_mes (%)']

print("Treinando o modelo...")
model_pipeline.fit(X_train, y_train)

# Avaliação
r2_score = model_pipeline.score(X_test, y_test)
best_alpha = model_pipeline.named_steps['regressor'].alpha_

print(f"--- Resultados ---")
print(f"Melhor Alpha (Regularização): {best_alpha}")
print(f"R² no Teste: {r2_score:.4f}")

Treinando o modelo...
--- Resultados ---
Melhor Alpha (Regularização): 100.0
R² no Teste: 0.5718


In [25]:
# --- 5. Interpretando o Encoding (Data Understanding Pós-Modelagem) ---

# É útil ver quais serventias tem as maiores taxas médias aprendidas pelo modelo
encoder = model_pipeline.named_steps['encoder']
# O encoder já foi treinado, podemos inspecionar o mapeamento (se necessário para relatório)
# Exemplo: transformando um dataframe dummy para ver os valores
print("\nExemplo de como o modelo 'vê' os dados agora:")
print(encoder.transform(X_test.head(3)))


Exemplo de como o modelo 'vê' os dados agora:
    ano  mes    comarca  serventia  Distribuídos_mes  Baixados_mes  \
0  2024    5  29.858562  42.586112               117            76   
1  2022   10  41.618414  42.215778                88             9   
2  2022    6  35.429505  42.215778               123            49   

   Pendentes_mes  
0             50  
1             24  
2             19  
